### Practical Comparison between Giraffe and DeepChess Model

#### Comparing how often Giraffe and DeepChess is able to successfully determine a "good" move (top 10 move recommendation by StockFish)

In [16]:
import torch
import torch.nn as nn
from stockfish import Stockfish
import chess
import itertools

In [7]:
class Giraffe(nn.Module):
    def __init__(self, global_dim, piece_dim, square_dim, global_nodes, piece_nodes, square_nodes, fc_nodes, dropout_rate):
        super(Giraffe, self).__init__()

        self.global_fc = nn.Sequential(
            nn.Linear(global_dim, global_nodes),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

        self.piece_fc = nn.Sequential(
            nn.Linear(piece_dim, piece_nodes),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

        self.square_fc = nn.Sequential(
            nn.Linear(square_dim, square_nodes),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

        self.combined_fc = nn.Sequential(
            nn.Linear(global_nodes+piece_nodes+square_nodes, fc_nodes),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(fc_nodes, 1),
            nn.Tanh()
        )

    def forward(self, X_g, X_p, X_s):
        x_g = self.global_fc(X_g)
        x_p = self.piece_fc(X_p)
        x_s = self.square_fc(X_s)
        x = torch.cat((x_g, x_p, x_s), dim=1)
        return self.combined_fc(x)


class Pos2Vec(nn.Module):
    def __init__(self, input_size, output_size):
        super(Pos2Vec, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, output_size),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(output_size, input_size),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

class FinalPos2Vec(nn.Module):
    def __init__(self, layer_sizes):
        super(FinalPos2Vec, self).__init__()
        self.encoders = nn.ModuleList([
            Pos2Vec(layer_sizes[i], layer_sizes[i+1]) for i in range(len(layer_sizes)-1)
        ])

    def forward(self, x):
        for encoder in self.encoders:
            x, _ = encoder(x) # only require encoded output
        return x # final encoded representation

class DeepChess(nn.Module):
    def __init__(self, pos2vec):
        super(DeepChess, self).__init__()
        self.pos2vec = pos2vec # Pos2Vec autoencoder
        self.comparator = nn.Sequential(
            nn.Linear(200, 400), # input is concatenation of two 100-d vectors
            nn.ReLU(),
            nn.Linear(400, 200),
            nn.ReLU(),
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Linear(100, 2), # binary classification btw the two positions
        )

    def forward(self, pos_a, pos_b):
        vec_a = self.pos2vec(pos_a) # encode positions
        vec_b = self.pos2vec(pos_b)
        combined = torch.cat((vec_a, vec_b), dim=1) # concatenate feature vectors
        output = self.comparator(combined) # output "better" position
        return output

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

giraffe = Giraffe(15, 208, 128, 12, 64, 24, 64, 0.4)
giraffe.load_state_dict(torch.load("./models/final_giraffe_model.pth", weights_only=True, map_location=torch.device('cpu')))
giraffe.to(device)

layer_sizes = [773, 600, 400, 200, 100]
pos2vec = FinalPos2Vec(layer_sizes)
deepchess_model_path = "./models/deepchess.pth"
deepchess = DeepChess(pos2vec)
deepchess.load_state_dict(torch.load(deepchess_model_path, weights_only=True, map_location=torch.device('cpu')))
deepchess.to(device)

DeepChess(
  (pos2vec): FinalPos2Vec(
    (encoders): ModuleList(
      (0): Pos2Vec(
        (encoder): Sequential(
          (0): Linear(in_features=773, out_features=600, bias=True)
          (1): ReLU()
        )
        (decoder): Sequential(
          (0): Linear(in_features=600, out_features=773, bias=True)
          (1): Sigmoid()
        )
      )
      (1): Pos2Vec(
        (encoder): Sequential(
          (0): Linear(in_features=600, out_features=400, bias=True)
          (1): ReLU()
        )
        (decoder): Sequential(
          (0): Linear(in_features=400, out_features=600, bias=True)
          (1): Sigmoid()
        )
      )
      (2): Pos2Vec(
        (encoder): Sequential(
          (0): Linear(in_features=400, out_features=200, bias=True)
          (1): ReLU()
        )
        (decoder): Sequential(
          (0): Linear(in_features=200, out_features=400, bias=True)
          (1): Sigmoid()
        )
      )
      (3): Pos2Vec(
        (encoder): Sequential(
    

In [ ]:
from utils.pgn_processor import pgn_to_fen, select_random_fens
from utils.giraffe_feature_extraction import extract_giraffe_features
from utils.giraffe_data_preparation import split_features
from utils.deepchess_feature_extraction import fen_to_bitboard

STOCKFISH_PATH = "C:\\Users\\User\\Downloads\\stockfish\\stockfish-windows-x86-64-avx2" # windows
# STOCKFISH_PATH = "/opt/homebrew/bin/stockfish" # mac
parameters = {
    "Threads": 12,
    "Hash": 256,
    "UCI_LimitStrength": True, # limit strength using Elo rating
    "UCI_Elo": 2800
}
# Initialize Stockfish with the given parameters
stockfish = Stockfish(path=STOCKFISH_PATH, parameters=parameters)

PGN_FOLDER = "./../strong_chess_players" # relative to code folder
fens = pgn_to_fen(PGN_FOLDER) # collect all unique FENs
random_fens = select_random_fens(fens, 100000) # select 100,000 random FENs

print("Done extracting games as FENs.")

giraffe_better_moves, deepchess_better_moves, same_move = 0, 0, 0
giraffe_correct, deepchess_correct = 0, 0
giraffe_correct_precision, deepchess_correct_precision = 0, 0
total_positions, skipped_positions = len(random_fens), 0

for i, fen in enumerate(random_fens):
    # Status update
    if (i+1) % 50000 == 0:
        print(f"Analysed {i+1}/{len(random_fens)} positions...")
    
    board = chess.Board(fen)
    # Prepare data for both models
    legal_moves = list(board.legal_moves) # get all legal moves
    if not legal_moves:  
        skipped_positions += 1
        total_positions -= 1
        continue

    giraffe_features = []
    deepchess_positions = {} # {move: bitboard}

    for move in legal_moves:
        board.push(move)
        # Giraffe
        giraffe_feature = extract_giraffe_features(board)
        giraffe_features.append(torch.tensor(giraffe_feature, dtype=torch.float32))
        # DeepChess
        deepchess_positions[move] = fen_to_bitboard(board.fen())
        board.pop()
        
    # Get best move from Giraffe
    giraffe.eval()
    X = torch.stack(giraffe_features).to(device)
    X_g, X_p, X_s = split_features(X)
    with torch.no_grad():
        scores = giraffe(X_g, X_p, X_s).squeeze()
    best_idx = torch.argmax(scores).item() if board.turn == chess.WHITE else torch.argmin(scores).item()
    giraffe_move = legal_moves[best_idx]

    # Get best move from DeepChess
    deepchess.eval()
    # Tournament-style selection: Each move competes against all others
    wins = {move: 0 for move in legal_moves}
    for move_a, move_b in itertools.combinations(legal_moves, 2):
        pos_a = torch.tensor(deepchess_positions[move_a], dtype=torch.float32).to(device).unsqueeze(0)
        pos_b = torch.tensor(deepchess_positions[move_b], dtype=torch.float32).to(device).unsqueeze(0)
        with torch.no_grad():
            output = deepchess(pos_a, pos_b).squeeze()
        if output[0] > output[1]: # DeepChess prefers move_a
            wins[move_a] += 1
        else: # DeepChess prefers move_b
            wins[move_b] += 1
    # Choose the move that won the most comparisons
    deepchess_move = max(wins, key=wins.get)

    # Evaluate Giraffe's move
    board.push(giraffe_move)
    stockfish.set_fen_position(board.fen())
    giraffe_eval = stockfish.get_evaluation()["value"]
    board.pop()

    # Evaluate DeepChess's move
    board.push(deepchess_move)
    stockfish.set_fen_position(board.fen())
    deepchess_eval = stockfish.get_evaluation()["value"]
    board.pop()

    giraffe_move_str = giraffe_move.uci()
    deepchess_move_str = deepchess_move.uci()

    # Determine which model made the better move
    if giraffe_move_str == deepchess_move_str or deepchess_eval == giraffe_eval:
        same_move += 1
    elif board.turn == chess.WHITE:
        giraffe_better_moves += int(giraffe_eval > deepchess_eval)
        deepchess_better_moves += int(deepchess_eval > giraffe_eval)
    else:
        giraffe_better_moves += int(giraffe_eval < deepchess_eval)
        deepchess_better_moves += int(deepchess_eval < giraffe_eval)
    
    
    # Get top 10 best moves from StockFish
    stockfish.set_fen_position(board.fen())
    stockfish.set_depth(1)
    stockfish_top_10 = {move_data['Move']: idx + 1 for idx, move_data in enumerate(stockfish.get_top_moves(10))}

    # If either moves from Giraffe or DeepChess matches a top 10 StockFish move, update their counter
    if giraffe_move_str in stockfish_top_10:
        giraffe_correct += 1
        giraffe_correct_precision += stockfish_top_10[giraffe_move_str]
    if deepchess_move_str in stockfish_top_10:    
        deepchess_correct += 1
        deepchess_correct_precision += stockfish_top_10[deepchess_move_str]

Error processing a game in Kasparov Garry 1973-1998 1878 Games.PGN: 'utf-8' codec can't decode byte 0x81 in position 6223: invalid start byte
Error processing a game in Keres Paul (EST, 1916-1975) 2324 Games expanded by Jan Malmstrom.PGN: 'utf-8' codec can't decode byte 0x81 in position 2962: invalid start byte
Error processing a game in Petrosian Tigran 1850 Games.PGN: 'utf-8' codec can't decode byte 0x81 in position 993: invalid start byte
Done extracting games as FENs.
Analysed 50000/100000 positions...
Analysed 100000/100000 positions...


In [47]:
print(f"Skipped positions (no legal moves): {skipped_positions}")

# Compare how often Giraffe and DeepChess make better moves
print(f"Giraffe made better moves than DeepChess in {giraffe_better_moves / total_positions:.2%} of positions.")
print(f"DeepChess made better moves than Giraffe in {deepchess_better_moves / total_positions:.2%}.")
print(f"Both models made the same move in {same_move / total_positions:.2%} of positions.")

# Accuracy of each model in predicting a "top 10 StockFish move"
print(f"Giraffe's accuracy in predicting a top 10 StockFish move: {giraffe_correct / total_positions:.2%}")
print(f"DeepChess's accuracy in predicting a top 10 StockFish move: {deepchess_correct / total_positions:.2%}")

# Average ranking of correctly predicted moves
if giraffe_correct > 0:
    print(f"Average StockFish ranking of Giraffe's correct moves: {giraffe_correct_precision / giraffe_correct:.2f}")
if deepchess_correct > 0:
    print(f"Average StockFish ranking of DeepChess's correct moves: {deepchess_correct_precision / deepchess_correct:.2f}")

Skipped positions (no legal moves): 20
Giraffe made better moves than DeepChess in 38.63% of positions.
DeepChess made better moves than Giraffe in 48.55%.
Both models made the same move in 12.82% of positions.
Giraffe's accuracy in predicting a top 10 StockFish move: 32.21%
DeepChess's accuracy in predicting a top 10 StockFish move: 37.00%
Average StockFish ranking of Giraffe's correct moves: 4.17
Average StockFish ranking of DeepChess's correct moves: 4.75
